In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.protobuf.internal.well_known_types import Timestamp
from sqlalchemy.dialects.mssql.information_schema import columns


Wszystkie pliki w katalogu `dane`

In [2]:
import glob
files=glob.glob(pathname="*.csv", root_dir='./dane')
dfs = [pd.read_csv('./dane/'+f,parse_dates = ['Date'],index_col='Date') for f in files]


In [3]:
dfs = []
for i, f in enumerate(files):
    df = pd.read_csv('./dane/'+f,parse_dates = ['Date'],index_col='Date')



    df = df.rename(columns={
        col: f"{f[:-4]}_{col}"
        for col in df.columns# optionally keep as metadata
    })
    # Drop static columns (lon/lat don't change per row)
    df = df.drop(columns=["longitude", "latitude"], errors="ignore")
    dfs.append(df)

# Merge all locations on time index
merged = pd.concat(dfs, axis=1)

C:\Users\klang\AppData\Local\Temp\ipykernel_22916\1957158990.py:16: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  merged = pd.concat(dfs, axis=1)


In [4]:
merged.dropna(inplace=True)

In [5]:
merged

,dolnośląskie_Avg_Air_Temp,dolnośląskie_Rel_Air_Humidity,dolnośląskie_Rainfall,dolnośląskie_Snowfall,dolnośląskie_Avg_Soil_Temp,dolnośląskie_Soil_Humidity,dolnośląskie_Wind_Speed,dolnośląskie_Dew_Point,dolnośląskie_Solar_Radiation,dolnośląskie_latidute,...,świętokrzyskie_Rel_Air_Humidity,świętokrzyskie_Rainfall,świętokrzyskie_Snowfall,świętokrzyskie_Avg_Soil_Temp,świętokrzyskie_Soil_Humidity,świętokrzyskie_Wind_Speed,świętokrzyskie_Dew_Point,świętokrzyskie_Solar_Radiation,świętokrzyskie_latidute,świętokrzyskie_longitude
Date,,,,,,,,,,,,,,,,,,,,,
2025-02-15 00:00:00,-3.24,89.0,0.00,0.0,0.00,21.0,2.08,-4.69,-6.0,51.065327,...,68.0,0.0,0.0,0.00,24.0,4.13,-10.19,36.0,50.667588,21.454414
2025-02-15 01:00:00,-2.62,90.0,0.00,0.0,0.00,21.0,1.71,-3.92,0.0,51.065327,...,75.0,0.0,0.0,0.00,24.0,4.16,-9.23,20.0,50.667588,21.454414
2025-02-15 02:00:00,-2.20,94.0,0.00,0.0,0.00,21.0,1.41,-3.02,-10.0,51.065327,...,79.0,0.0,0.0,0.00,25.0,3.17,-8.34,6.0,50.667588,21.454414
2025-02-15 03:00:00,-2.17,90.0,0.00,0.0,0.00,21.0,1.21,-3.52,-5.0,51.065327,...,83.0,0.0,0.0,0.00,25.0,2.97,-7.69,5.0,50.667588,21.454414
2025-02-15 04:00:00,-2.50,89.0,0.00,0.0,0.00,21.0,1.33,-4.02,-2.0,51.065327,...,85.0,0.0,0.0,0.00,25.0,2.71,-7.53,2.0,50.667588,21.454414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-13 19:00:00,11.29,64.0,0.00,0.0,13.11,22.0,2.46,4.77,-15.0,51.065327,...,61.0,0.0,0.0,14.41,21.0,2.23,3.74,22.0,50.667588,21.454414
2026-05-13 20:00:00,9.83,68.0,0.18,0.0,13.08,22.0,1.79,4.41,-8.0,51.065327,...,69.0,0.0,0.0,13.80,21.0,1.55,4.04,-21.0,50.667588,21.454414
2026-05-13 21:00:00,8.81,87.0,0.15,0.0,12.54,22.0,1.24,6.81,-13.0,51.065327,...,77.0,0.0,0.0,12.67,21.0,1.59,3.94,-23.0,50.667588,21.454414


In [6]:
for i in range(merged.shape[0]-1):
    if  not (merged.index[i+1]-merged.index[i])==pd.Timedelta('0 days 01:00:00'):
        print(i)


In [7]:
production_df = pd.read_excel("./dane/fw_pv.xlsx", parse_dates=["Data"])


In [8]:
production_df

,Data,Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal,Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal
0,2022-10-09 00:00:00+00:00,3657.888,0.0
1,2022-10-09 01:00:00+00:00,3638.238,0.0
2,2022-10-09 02:00:00+00:00,3457.525,0.0
3,2022-10-09 03:00:00+00:00,3491.475,0.0
4,2022-10-09 04:00:00+00:00,3378.375,0.0
...,...,...,...
31551,2026-05-15 19:00:00+00:00,NaN,NaN
31552,2026-05-15 20:00:00+00:00,NaN,NaN
31553,2026-05-15 21:00:00+00:00,NaN,NaN
31554,2026-05-15 22:00:00+00:00,NaN,NaN


Przycięte jest do 13 maja bo były wartości Na

In [9]:
production_df = production_df[production_df["Data"]<"2026-05-13 07:00:00"]
production_df['Data']=pd.to_datetime(production_df['Data']).dt.strftime("%Y-%m-%d %H:%M:%S")

production_df["Data"] = pd.to_datetime(production_df["Data"])

In [10]:
production_df

,Data,Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal,Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal
0,2022-10-09 00:00:00,3657.88800,0.0000
1,2022-10-09 01:00:00,3638.23800,0.0000
2,2022-10-09 02:00:00,3457.52500,0.0000
3,2022-10-09 03:00:00,3491.47500,0.0000
4,2022-10-09 04:00:00,3378.37500,0.0000
...,...,...,...
31486,2026-05-13 02:00:00,3361.20400,0.0000
31487,2026-05-13 03:00:00,3373.31850,0.0000
31488,2026-05-13 04:00:00,3375.73425,72.2925
31489,2026-05-13 05:00:00,3095.13100,466.3460


In [11]:
production_df.interpolate(inplace=True)

,Data,Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal,Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal
0,2022-10-09 00:00:00,3657.88800,0.0000
1,2022-10-09 01:00:00,3638.23800,0.0000
2,2022-10-09 02:00:00,3457.52500,0.0000
3,2022-10-09 03:00:00,3491.47500,0.0000
4,2022-10-09 04:00:00,3378.37500,0.0000
...,...,...,...
31486,2026-05-13 02:00:00,3361.20400,0.0000
31487,2026-05-13 03:00:00,3373.31850,0.0000
31488,2026-05-13 04:00:00,3375.73425,72.2925
31489,2026-05-13 05:00:00,3095.13100,466.3460


In [12]:
production_df = production_df.set_index("Data").sort_index()

In [13]:
lista_brakow = []
for i in range(production_df.shape[0]-1):
    if  not (production_df.index[i+1]-production_df.index[i])==pd.Timedelta('0 days 01:00:00'):
        lista_brakow.append(i)

W środku danych zdażały się braki kilka razy to są zastąpione interpolacją

In [14]:
lista_brakow

[4033, 12936, 21671, 30406]

In [15]:
production_df.columns

Index(['Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal', 'Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal'], dtype='str')

In [16]:
for i in lista_brakow:
    new_date = pd.to_datetime(production_df.index[i]+pd.Timedelta(hours=1))
    production_df.loc[new_date]=[np.float64(np.nan),np.float64(np.nan)]

In [17]:
production_df.tail()

,Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal,Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal
Data,,
2026-05-13 06:00:00,2770.41,1024.788
2023-03-26 02:00:00,NaN,NaN
2024-03-31 02:00:00,NaN,NaN
2025-03-30 02:00:00,NaN,NaN
2026-03-29 02:00:00,NaN,NaN


In [18]:
production_df.sort_index(inplace=True)

In [19]:
production_df.interpolate(inplace=True)

,Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal,Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal
Data,,
2022-10-09 00:00:00,3657.88800,0.0000
2022-10-09 01:00:00,3638.23800,0.0000
2022-10-09 02:00:00,3457.52500,0.0000
2022-10-09 03:00:00,3491.47500,0.0000
2022-10-09 04:00:00,3378.37500,0.0000
...,...,...
2026-05-13 02:00:00,3361.20400,0.0000
2026-05-13 03:00:00,3373.31850,0.0000
2026-05-13 04:00:00,3375.73425,72.2925


Dodane kolumny z czasem i

In [20]:
production_df['hour'] = pd.DatetimeIndex(production_df.index).hour
production_df['day'] = pd.DatetimeIndex(production_df.index).day
production_df['day_of_the_year'] = pd.DatetimeIndex(production_df.index).dayofyear
production_df['month']= pd.DatetimeIndex(production_df.index).month
production_df['quarter'] = pd.DatetimeIndex(production_df.index).quarter

In [21]:
production_df.columns

Index(['Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal',
       'Sumaryczna generacja Źródeł Fotowoltaicznych his-wlk-cal', 'hour',
       'day', 'day_of_the_year', 'month', 'quarter'],
      dtype='str')

In [22]:
wind_production=production_df.iloc[:,[True,False,True,True,True,True,True]]

In [23]:
wind_production=wind_production.rename(columns=
                    {'Sumaryczna generacja Źródeł Wiatrowych his-wlk-cal':'fw_production'})
wind_production

,fw_production,hour,day,day_of_the_year,month,quarter
Data,,,,,,
2022-10-09 00:00:00,3657.88800,0,9,282,10,4
2022-10-09 01:00:00,3638.23800,1,9,282,10,4
2022-10-09 02:00:00,3457.52500,2,9,282,10,4
2022-10-09 03:00:00,3491.47500,3,9,282,10,4
2022-10-09 04:00:00,3378.37500,4,9,282,10,4
...,...,...,...,...,...,...
2026-05-13 02:00:00,3361.20400,2,13,133,5,2
2026-05-13 03:00:00,3373.31850,3,13,133,5,2
2026-05-13 04:00:00,3375.73425,4,13,133,5,2


Czasowe kolumny konwertowane na sin/cos aby były cykliczne

In [24]:
def add_cyclical_encoding(df):
    """
    Converts datetime columns to sin/cos pairs.
    Drops the original integer columns afterward.
    """
    cycles = {
        "hour"    : 24,
        "day"     : 31,
        "day_of_the_year"    : 356,
        "month"   : 12,
        "quarter" : 4,
    }
    for col, period in cycles.items():
        if col in df.columns:
            df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
            df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)
            df.drop(columns=[col], inplace=True)  # drop raw integer
    return df

In [25]:
wind_production=add_cyclical_encoding(wind_production)

In [26]:
df_full = merged.join(wind_production,how="inner")

In [27]:
df_full.shape

(10855, 187)

In [28]:
df_full.to_csv("dane_wind.csv")

Tutaj zamiast nominalnych wartości są godzinowe różnice, w produkcji.

In [37]:
newone = [df_full['fw_production'].iloc[i+1]-df_full['fw_production'].iloc[i]
          for i in range(len(df_full['fw_production'])-1)]

In [44]:
newone=[0]+newone

In [47]:
df_full['fw_prod_diff']=newone

C:\Users\klang\AppData\Local\Temp\ipykernel_22916\1408469608.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_full['fw_prod_diff']=l


In [48]:
df_full.head()

,dolnośląskie_Avg_Air_Temp,dolnośląskie_Rel_Air_Humidity,dolnośląskie_Rainfall,dolnośląskie_Snowfall,dolnośląskie_Avg_Soil_Temp,dolnośląskie_Soil_Humidity,dolnośląskie_Wind_Speed,dolnośląskie_Dew_Point,dolnośląskie_Solar_Radiation,dolnośląskie_latidute,...,hour_cos,day_sin,day_cos,day_of_the_year_sin,day_of_the_year_cos,month_sin,month_cos,quarter_sin,quarter_cos,fw_prod_diff
Date,,,,,,,,,,,,,,,,,,,,,
2025-02-15 00:00:00,-3.24,89.0,0.0,0.0,0.0,21.0,2.08,-4.69,-6.0,51.065327,...,1.000000,0.101168,-0.994869,0.725577,0.688141,0.866025,0.5,1.0,6.123234e-17,0.00000
2025-02-15 01:00:00,-2.62,90.0,0.0,0.0,0.0,21.0,1.71,-3.92,0.0,51.065327,...,0.965926,0.101168,-0.994869,0.725577,0.688141,0.866025,0.5,1.0,6.123234e-17,85.43475
2025-02-15 02:00:00,-2.20,94.0,0.0,0.0,0.0,21.0,1.41,-3.02,-10.0,51.065327,...,0.866025,0.101168,-0.994869,0.725577,0.688141,0.866025,0.5,1.0,6.123234e-17,-33.47825
2025-02-15 03:00:00,-2.17,90.0,0.0,0.0,0.0,21.0,1.21,-3.52,-5.0,51.065327,...,0.707107,0.101168,-0.994869,0.725577,0.688141,0.866025,0.5,1.0,6.123234e-17,-65.99075
2025-02-15 04:00:00,-2.50,89.0,0.0,0.0,0.0,21.0,1.33,-4.02,-2.0,51.065327,...,0.500000,0.101168,-0.994869,0.725577,0.688141,0.866025,0.5,1.0,6.123234e-17,-139.95725


In [50]:
df_full[['fw_prod_diff','fw_production']]

,fw_prod_diff,fw_production
Date,,
2025-02-15 00:00:00,0.00000,1609.97075
2025-02-15 01:00:00,85.43475,1695.40550
2025-02-15 02:00:00,-33.47825,1661.92725
2025-02-15 03:00:00,-65.99075,1595.93650
2025-02-15 04:00:00,-139.95725,1455.97925
...,...,...
2026-05-13 02:00:00,-76.52550,3361.20400
2026-05-13 03:00:00,12.11450,3373.31850
2026-05-13 04:00:00,2.41575,3375.73425


In [51]:
df_full=df_full.drop(columns=["fw_production"])

In [53]:
df_full.to_csv("dane_wind_diff.csv")